# S2.11 — Partitioning
**Date completed:** September 2026  
**Status:** In Progress  
**Interview covered:** Q11 — What is a partition in Spark?

# S2.11 — Spark Partitioning Deep Dive
### Topic: Partitioning, Parallelism, Coalesce, Repartition, partitionBy and Partition Pruning

---

## 1. What is partitioning in Spark?

Partitioning means dividing a large dataset into smaller logical portions so Spark can process data across multiple tasks.

For example:

10 million rows → 8 partitions → approximately 1.25 million rows per partition.

Each partition is processed by one task within a stage. Multiple tasks can execute simultaneously when sufficient CPU cores are available.

**Key principle:** Partitioning enables parallel processing, but having more partitions does not automatically guarantee better performance.

---

## 2. How many types of partitioning are there?

For our learning, separate partitioning into TWO major categories:

| Category | Purpose |
|---|---|
| Execution/DataFrame Partitioning | Controls how Spark distributes and processes data |
| Storage Partitioning | Controls how data files are organised on storage |

### A. Execution/DataFrame Partitioning

| Method | Purpose | Shuffle |
|---|---|---|
| Automatic partitioning | Spark determines partitions based on source and configuration | Depends on operation |
| coalesce(n) | Primarily reduces partition count | Normally no full shuffle |
| repartition(n) | Redistributes data into n partitions | Yes |
| repartition(n, column) | Redistributes data using partitioning columns, generally hash-based | Yes |
| Shuffle partitioning | Redistributes intermediate data for joins and aggregations | Yes |

### B. Storage Partitioning

| Method | Purpose |
|---|---|
| write.partitionBy(column) | Organises written data into directories based on column values |
| Partition Pruning | Skips unnecessary storage partitions during reading |

Important: Storage partitioning and Spark execution partitioning are NOT the same thing.

---

## 3. Automatic Partitioning

Example:

```python
df = spark.range(0, 10000000)
```

We did not manually specify any partition count.

Spark automatically determines the initial partitions based on the source, available compute and configuration.

To check the observed non-empty partition count:

```python
df.select(F.spark_partition_id()).distinct().count()
```

Our practical result:

Default partitions = 8

Note: Shuffle operations may introduce different partition counts later in the execution plan.

---

## 4. coalesce() vs repartition()

### coalesce()

```python
df.coalesce(1)
df.coalesce(2)
```

- Primarily used to reduce partitions.
- Normally avoids a full shuffle.
- Reducing partitions too much can decrease parallelism.

### repartition()

```python
df.repartition(8)
df.repartition(16)
df.repartition(500)
```

- Can increase or decrease the partition count.
- Redistributes data through a shuffle.
- Creates additional data-movement and scheduling costs.

**Interview point:**

Coalesce reduces partitions with minimal redistribution, whereas repartition performs a shuffle to redistribute data.

---

## 5. Our Practical Experiment — 10 Million Rows

| Scenario | Partitions | Execution Time |
|---|---:|---:|
| Under-parallelism | 1 | 1.3048 seconds |
| Many partitions | 500 | 2.3609 seconds |
| Eight partitions | 8 | 0.7127 seconds |

### Observations

1. One partition restricted input-stage parallelism.
2. Five hundred partitions introduced additional shuffle and task overhead.
3. Eight partitions performed fastest in this particular execution.
4. All three scenarios returned the same aggregation result: 4.995E9.

**Important:** Eight partitions are not universally optimal. Our experiment also compared coalesce and repartition, which have different execution costs.

Always measure performance instead of assuming a partition count is optimal.

---

## 6. What is Parallelism?

Parallelism means executing multiple tasks simultaneously.

Example:

8 CPU cores + 8 independent tasks = potentially 8 concurrent tasks.

8 CPU cores + 1 task = only one core processes that task.

16 partitions + 4 available CPU cores = approximately 4 concurrent tasks, assuming one core per task.

Remaining tasks wait until processing resources become available.

The executor is the Spark process running tasks, and an executor may have multiple CPU cores.

**Important:** Partitions determine the number of tasks in a stage. Available processing resources determine how many tasks can run concurrently.

---

## 7. Storage Partitioning — partitionBy()

Example:

```python
df_orders.write \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .parquet(output_path)
```

Spark organises data into date-based directories:

orders/
- order_date=2026-09-24/
- order_date=2026-09-25/
- order_date=2026-09-26/

Each directory contains the data files written for that date.

The number of directories does not necessarily equal the number of files or Spark execution partitions.

### Partition Pruning

```python
spark.read.parquet(output_path) \
    .filter(F.col("order_date") == "2026-09-26")
```

Spark can skip irrelevant date directories and read only the required storage partition.

We can verify this using:

```python
df.explain(True)
```

Look for `PartitionFilters` in the physical execution plan.

---

# 8. Enterprise Question: What if we have trillions of records?

**Do not decide the partition count using row count alone.**

One trillion records might occupy very different storage sizes depending on row width, compression, data types and file format.

Partition planning should consider:

| Factor | Why it matters |
|---|---|
| Total data size in GB/TB/PB | Determines overall processing volume |
| Average row size | Helps estimate data volume |
| Available CPU cores | Determines potential task parallelism |
| Memory availability | Affects spilling and task execution |
| Data distribution | Helps identify skew |
| Transformation type | Determines shuffle and aggregation requirements |
| File sizes | Helps avoid excessive small-file overhead |
| Query patterns | Helps determine storage partitioning |

### Step 1 — Understand the data

Before configuring partitions, identify:

- Total data size.
- Number of records.
- Average file size.
- Existing input partition count.
- Available executor cores.
- Query and transformation requirements.

### Step 2 — Estimate execution partitions

A useful initial heuristic for many file-processing workloads is to target approximately 128–256 MB of input data per task.

Conceptual formula:

Estimated input partitions = Total input data size / Target data size per partition

This is a starting point, not a fixed Spark rule. Different workloads may benefit from smaller or larger partitions.

Also consider available CPU cores and how many waves of concurrent tasks are required.

### Step 3 — Identify data skew

Suppose a city-wise dataset contains:

Delhi: 70% of records  
Mumbai: 10%  
Pune: 10%  
Chennai: 10%

Some tasks may receive significantly more work than others.

Even with a suitable overall partition count, skew can create slow tasks and reduce performance.

We should inspect task duration, shuffle metrics, spill and partition distribution.

### Step 4 — Plan storage partitioning separately

For very large historical datasets, date-based storage partitioning may be useful.

Example:

```python
df.write \
    .partitionBy("order_date") \
    .format("delta") \
    .save(output_path)
```

However, storage partitioning must match query patterns and data volume.

Avoid partitioning by high-cardinality values such as individual order IDs or transaction IDs, which can create excessive directories and small files.

### Step 5 — Measure and tune

Review the Spark execution plan and Query Profile.

Check:

- Task execution duration.
- Number of tasks.
- Shuffle read/write.
- Memory usage and disk spill.
- Data skew.
- Scan volume.
- File sizes.
- Total execution time.

Adaptive Query Execution (AQE) can also optimise shuffle partitioning dynamically, including coalescing shuffle partitions.

In Databricks Serverless, Spark may use automatic shuffle partition management (`spark.sql.shuffle.partitions = auto`).

Do not manually tune parameters without checking whether the platform already manages them.

---

## 9. Final Decision Framework

| Situation | What should we investigate? |
|---|---|
| Too few processing partitions | Increasing parallelism |
| Excessive small partitions | Reducing partition/task overhead |
| Expensive shuffle | Shuffle metrics, keys and data movement |
| Uneven task durations | Data skew |
| Queries repeatedly filter by date | Storage partitioning and partition pruning |
| Large numbers of tiny files | File compaction and layout optimisation |
| High-volume repeated processing | Incremental processing and appropriate data layout |

There is no universally correct partition count.

For trillion-row datasets, first calculate an approximate partition requirement from data volume and workload characteristics, then benchmark using available resources and actual execution metrics.

---

## 10. Interview Summary

**Q: What is partitioning in Spark?**

Partitioning divides distributed data into smaller logical units that Spark tasks can process in parallel.

**Q: What is the difference between coalesce and repartition?**

Coalesce primarily reduces partitions and normally avoids a full shuffle. Repartition redistributes data using a shuffle and can increase or decrease partition count.

**Q: What is partitionBy?**

It organises written files based on column values to support efficient storage organisation and partition pruning.

**Q: How would you decide partition count for one trillion records?**

I would evaluate actual data size, input file sizes, compute resources, transformations and data distribution. I would estimate an initial partition count using data volume per task, benchmark execution, analyse shuffle and task metrics, and adjust accordingly.

I would also design storage partitioning separately based on query patterns and incremental processing requirements.

### Final Learning Principle

**Partitioning is not about creating the maximum possible number of partitions. It is about balancing parallelism, data movement, memory utilisation and execution overhead to process data efficiently.**

In [0]:
# ============================================================
# Cell 2 — Partitioning: Prove the Goldilocks rule
# Goal: See what happens with too few, too many, just right
# ============================================================

import pyspark.sql.functions as F
import time

print("=== PARTITIONING DEEP DIVE ===")
print()

# Create 10M row dataset
df = spark.range(0, 10000000)
df = df.withColumn("value", (F.col("id") % 1000).cast("double"))

# Check partition count using spark_partition_id
default_partitions = df.select(F.spark_partition_id()).distinct().count()
print(f"Default partitions: {default_partitions}")
print()

# --- Scenario 1: Too few partitions ---
print("--- Scenario 1: Too few partitions (under-parallelism) ---")
df_few = df.coalesce(1)
few_partitions = df_few.select(F.spark_partition_id()).distinct().count()
print(f"Partitions: {few_partitions}")

start = time.time()
df_few.agg(F.sum("value")).show()
end = time.time()
print(f"Time with 1 partition: {round(end-start, 4)} seconds")
print()

# --- Scenario 2: Too many partitions ---
print("--- Scenario 2: Too many partitions (over-partitioning) ---")
df_many = df.repartition(500)
many_partitions = df_many.select(F.spark_partition_id()).distinct().count()
print(f"Partitions: {many_partitions}")
start = time.time()
df_many.agg(F.sum("value")).show()
end = time.time()
print(f"Time with 500 partitions: {round(end-start, 4)} seconds")
print()

# --- Scenario 3: Just right ---
print("--- Scenario 3: Just right (2-3x cores) ---")
df_right = df.repartition(8)
right_partitions = df_right.select(F.spark_partition_id()).distinct().count()
print(f"Partitions: {right_partitions}")
start = time.time()
df_right.agg(F.sum("value")).show()
end = time.time()
print(f"Time with 8 partitions: {round(end-start, 4)} seconds")

In [0]:
# ============================================================
# Cell 3 — Understanding Storage Partitioning (partitionBy)
#
# Goal:
# 1. Understand how partitionBy organises data on disk
# 2. Identify logical partition values
# 3. Understand partition pruning
#
# Note:
# This is a conceptual demonstration only.
# No physical partitioned files are created in this cell.
# Actual storage implementation will be covered in S8.
# ============================================================

import pyspark.sql.functions as F

print("=== STORAGE PARTITIONING DEEP DIVE ===")
print()

# ------------------------------------------------------------
# Step 1 — Create sample orders
# ------------------------------------------------------------
df_orders = spark.createDataFrame([
    (1001, "2026-09-24", "Delhi",   2500.0),
    (1002, "2026-09-24", "Mumbai",  1800.0),
    (1003, "2026-09-25", "Delhi",   3200.0),
    (1004, "2026-09-25", "Pune",     950.0),
    (1005, "2026-09-26", "Chennai", 4100.0),
    (1006, "2026-09-26", "Delhi",    750.0),
], ["order_id", "order_date", "city", "order_value"])

# Convert string to DateType — enterprise best practice
df_orders = df_orders.withColumn("order_date", F.to_date("order_date"))

print("Sample dataset:")
df_orders.show()

# ------------------------------------------------------------
# Step 2 — Understand logical partition values
# groupBy used ONLY to count rows per date
# Does NOT create physical storage partitions
# ------------------------------------------------------------
print("=== RECORD DISTRIBUTION BY DATE ===")
df_orders.groupBy("order_date") \
         .agg(F.count("*").alias("record_count")) \
         .orderBy("order_date") \
         .show()

# ------------------------------------------------------------
# Step 3 — Expected storage structure (illustrative only)
# ------------------------------------------------------------
print("=== EXPECTED STORAGE PARTITION STRUCTURE ===")
print("""
orders/
    |-- order_date=2026-09-24/
    |       |-- part-00000.parquet  (2 records)
    |
    |-- order_date=2026-09-25/
    |       |-- part-00000.parquet  (2 records)
    |
    |-- order_date=2026-09-26/
            |-- part-00000.parquet  (2 records)
""")

# ------------------------------------------------------------
# Step 4 — Enterprise implementation syntax
# Execute in S8 using authorised ADLS Gen2 storage
# ------------------------------------------------------------
print("=== ENTERPRISE IMPLEMENTATION (S8) ===")
print("""
# Write partitioned by date:
df_orders.write \\
    .mode("overwrite") \\
    .partitionBy("order_date") \\
    .parquet("abfss://bronze@retailpulse.dfs.core.windows.net/orders/")

# Read with partition pruning:
spark.read \\
    .parquet(".../orders/") \\
    .filter(F.col("order_date") == "2026-09-26")
# → Spark skips Sept 24 and Sept 25 folders entirely
# → Only reads Sept 26 partition
# → 1000x faster on billion-row tables
""")

# ------------------------------------------------------------
# Step 5 — Partition pruning summary
# Full verification with Query Profile in S8
# ------------------------------------------------------------
print("=== PARTITION PRUNING RULE ===")
print("Without partitioning : Spark reads ALL rows → filters in memory")
print("With partitioning    : Spark skips irrelevant folders → reads less")
print("Enterprise standard  : Always partition large tables by date or region")
print("Full hands-on        : S8 with real ADLS Gen2 storage")

## Key Takeaways — S2.11 Partitioning

## What is a Partition?
A partition is one chunk of your data processed by one executor.
All partitions are processed SIMULTANEOUSLY = parallelism.

## The Goldilocks Rule — Proved with Real Numbers
| Partitions | Time | Problem |
|-----------|------|---------|
| 1 | 1.3048s | 7 executors idle — under-parallelism |
| 500 | 2.3609s | Scheduling overhead > processing time |
| 8 ✅ | 0.7127s | All executors busy — just right |

**Rule: Optimal = 2-3x number of executor cores**

## 3 Types of Partitioning
| Type | Used by | How |
|------|---------|-----|
| Default | spark.range(), file reads | Based on cores or file size |
| Hash | groupBy(), join() | Same key → same partition |
| Range | orderBy() | Sorted ranges across partitions |

## Storage Partitioning (partitionBy)

Creates folder per partition value:
- order_date=2026-09-24/ → all Sept 24 rows
- order_date=2026-09-25/ → all Sept 25 rows

## Partition Pruning
Reading partitioned storage with a filter:
Spark skips irrelevant folders entirely.
1 billion rows partitioned by date → read 1 day = 1/365 of data.

## Enterprise Standards
- Partition large tables by date or region
- Never partition by high-cardinality column (order_id = bad)
- Target 128MB-256MB per partition
- Full implementation in S8 with ADLS Gen2